# ISR-RL-DMPC Learning Curves Visualization

Post-training analysis notebook for training metrics produced by `scripts/train_agent.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
def load_training_data(log_dir):
    """Load training metrics from CSV and JSON files."""
    log_path = Path(log_dir)
    csv_files = sorted(log_path.glob('*_metrics_*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No metrics CSV found in {log_path}')
    df = pd.read_csv(csv_files[-1])
    stats_file = log_path / 'training_stats.json'
    stats = None
    if stats_file.exists():
        with open(stats_file, 'r') as f:
            stats = json.load(f)
    return df, stats

# Update the path below to point at your training run directory
log_dir = '../data/training_logs'  
# If there are multiple runs, pick the latest sub-directory:
run_dirs = sorted(Path(log_dir).iterdir()) if Path(log_dir).exists() else []
if run_dirs:
    log_dir = str(run_dirs[-1])
    df, stats = load_training_data(log_dir)
    print(f'Loaded {len(df)} episodes from {log_dir}')
    print(f'Columns: {df.columns.tolist()}')
    display(df.head())
else:
    print('No training data found. Run scripts/train_agent.py first.')
    df = None

In [ ]:
def plot_episode_rewards(df, window=100):
    """Plot episode rewards with rolling mean."""
    fig, ax = plt.subplots(figsize=(14, 6))
    episodes = df['episode'].values
    rewards = df['reward'].values
    ax.plot(episodes, rewards, alpha=0.3, color='blue', label='Episode Reward')
    rolling = pd.Series(rewards).rolling(window=window, min_periods=1).mean()
    ax.plot(episodes, rolling, color='darkblue', linewidth=2,
            label=f'Rolling Mean ({window} ep)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Reward')
    ax.set_title('Training Progress: Episode Rewards', fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

if df is not None:
    plot_episode_rewards(df)
    plt.show()

In [ ]:
def plot_coverage_efficiency(df, window=50):
    """Plot coverage efficiency over training."""
    fig, ax = plt.subplots(figsize=(14, 6))
    episodes = df['episode'].values
    coverage = df['coverage'].values * 100
    ax.plot(episodes, coverage, alpha=0.3, color='green', label='Coverage %')
    rolling = pd.Series(coverage).rolling(window=window, min_periods=1).mean()
    ax.plot(episodes, rolling, color='darkgreen', linewidth=2,
            label=f'Rolling Mean ({window} ep)')
    ax.axhline(y=95, color='red', linestyle='--', linewidth=2, label='Target (95%)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Coverage Efficiency (%)')
    ax.set_title('Training Progress: Coverage Efficiency', fontweight='bold')
    ax.set_ylim([0, 100])
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

if df is not None:
    plot_coverage_efficiency(df)
    plt.show()

In [ ]:
def plot_energy_efficiency(df, window=50):
    """Plot energy efficiency (coverage per 100Wh)."""
    fig, ax = plt.subplots(figsize=(14, 6))
    episodes = df['episode'].values
    energy_eff = df['energy_eff'].values
    ax.plot(episodes, energy_eff, alpha=0.3, color='orange', label='Energy Efficiency')
    rolling = pd.Series(energy_eff).rolling(window=window, min_periods=1).mean()
    ax.plot(episodes, rolling, color='darkorange', linewidth=2,
            label=f'Rolling Mean ({window} ep)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Coverage per 100Wh (%)')
    ax.set_title('Training Progress: Energy Efficiency', fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

if df is not None:
    plot_energy_efficiency(df)
    plt.show()

In [ ]:
def plot_collision_rate(df, window=100):
    """Plot collision rate over training."""
    fig, ax = plt.subplots(figsize=(14, 6))
    episodes = df['episode'].values
    collisions = df['collisions'].values
    rolling = pd.Series(collisions).rolling(window=window, min_periods=1).mean()
    ax.plot(episodes, rolling, color='red', linewidth=2,
            label=f'Collision Rate ({window} ep avg)')
    ax.fill_between(episodes, 0, rolling, alpha=0.3, color='red')
    ax.axhline(y=0, color='green', linestyle='--', linewidth=2, label='Target (0)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Collisions per Episode')
    ax.set_title('Training Progress: Collision Rate', fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

if df is not None:
    plot_collision_rate(df)
    plt.show()

In [ ]:
def plot_loss_curves(df, window=50):
    """Plot critic and actor loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    episodes = df['episode'].values
    # Critic loss
    critic = pd.Series(df['critic_loss'].values).rolling(window=window, min_periods=1).mean()
    ax1.plot(episodes, critic, color='purple', linewidth=2)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Critic Loss')
    ax1.set_title('Value Network Loss', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    # Actor loss
    actor = pd.Series(df['actor_loss'].values).rolling(window=window, min_periods=1).mean()
    ax2.plot(episodes, actor, color='teal', linewidth=2)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Actor Loss')
    ax2.set_title('Policy Network Loss', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

if df is not None:
    plot_loss_curves(df)
    plt.show()

In [ ]:
def plot_dashboard(df, window=100):
    """Create comprehensive training dashboard."""
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)
    episodes = df['episode'].values

    # 1. Episode Reward
    ax1 = fig.add_subplot(gs[0, 0])
    rolling_r = pd.Series(df['reward'].values).rolling(window=window, min_periods=1).mean()
    ax1.plot(episodes, rolling_r, color='blue', linewidth=2)
    ax1.set_title('Episode Reward')
    ax1.grid(True, alpha=0.3)

    # 2. Coverage
    ax2 = fig.add_subplot(gs[0, 1])
    rolling_c = pd.Series(df['coverage'].values * 100).rolling(window=window, min_periods=1).mean()
    ax2.plot(episodes, rolling_c, color='green', linewidth=2)
    ax2.axhline(y=95, color='red', linestyle='--')
    ax2.set_title('Coverage (%)')
    ax2.grid(True, alpha=0.3)

    # 3. Collisions
    ax3 = fig.add_subplot(gs[1, 0])
    rolling_col = pd.Series(df['collisions'].values).rolling(window=window, min_periods=1).mean()
    ax3.plot(episodes, rolling_col, color='red', linewidth=2)
    ax3.set_title('Collisions per Episode')
    ax3.grid(True, alpha=0.3)

    # 4. Energy Efficiency
    ax4 = fig.add_subplot(gs[1, 1])
    rolling_e = pd.Series(df['energy_eff'].values).rolling(window=window, min_periods=1).mean()
    ax4.plot(episodes, rolling_e, color='orange', linewidth=2)
    ax4.set_title('Energy Efficiency')
    ax4.grid(True, alpha=0.3)

    # 5. Critic Loss
    ax5 = fig.add_subplot(gs[2, 0])
    rolling_cl = pd.Series(df['critic_loss'].values).rolling(window=window, min_periods=1).mean()
    ax5.plot(episodes, rolling_cl, color='purple', linewidth=2)
    ax5.set_title('Critic Loss')
    ax5.grid(True, alpha=0.3)

    # 6. Actor Loss
    ax6 = fig.add_subplot(gs[2, 1])
    rolling_al = pd.Series(df['actor_loss'].values).rolling(window=window, min_periods=1).mean()
    ax6.plot(episodes, rolling_al, color='teal', linewidth=2)
    ax6.set_title('Actor Loss')
    ax6.grid(True, alpha=0.3)

    fig.suptitle('ISR-RL-DMPC Training Dashboard', fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    return fig

if df is not None:
    plot_dashboard(df)
    plt.show()